# Bellabeat Fitness Tracker — Consumer Behavior Analysis
**Business task:** Analyze Fitbit smart-device data to uncover activity, sleep, and usage trends, and recommend product/marketing strategies for Bellabeat's wellness products.

**Data:** "FitBit Fitness Tracker Data" (CC0 Public Domain, via Kaggle), 30+ consenting Fitbit users, April–May 2016. Files used: `dailyActivity_merged.csv`, `sleepDay_merged.csv`, `hourlySteps_merged.csv`, `hourlyIntensities_merged.csv`.

*Portfolio rebuild of the 2024 Google Data Analytics capstone analysis; original coursework files were lost.*


## 1. Load and inspect the data

In [1]:
import pandas as pd
import numpy as np

daily = pd.read_csv('data/dailyActivity_merged.csv')
sleep = pd.read_csv('data/sleepDay_merged.csv')
print(f"dailyActivity rows: {len(daily)}, users: {daily['Id'].nunique()}")
print(f"sleepDay rows: {len(sleep)}, users: {sleep['Id'].nunique()}")
print(daily.dtypes)

dailyActivity rows: 940, users: 33
sleepDay rows: 413, users: 24


## 2. Clean the data
- Parse date columns to datetime
- Drop duplicate rows
- Remove non-wear days: rows with 0 steps and a (near-)full day of sedentary minutes — the tracker wasn't worn, not a real zero-activity day

In [1]:
daily['ActivityDate'] = pd.to_datetime(daily['ActivityDate'], format='%m/%d/%Y')
sleep['SleepDay'] = pd.to_datetime(sleep['SleepDay'], format='%m/%d/%Y %I:%M:%S %p')
daily = daily.drop_duplicates()
sleep = sleep.drop_duplicates()

nonwear = (daily['TotalSteps'] == 0) & (daily['SedentaryMinutes'] >= 1400)
print(f"Non-wear days removed: {nonwear.sum()} (affected {daily.loc[nonwear, 'Id'].nunique()} users)")
daily = daily[~nonwear].copy()
print(f"Clean daily rows: {len(daily)}, users: {daily['Id'].nunique()}")

daily['TotalActiveMinutes'] = (daily['LightlyActiveMinutes'] + daily['FairlyActiveMinutes']
                               + daily['VeryActiveMinutes'])

Non-wear days removed: 73 (affected 15 users)
Clean daily rows: 867, users: 33


## 3. Headline activity & sleep statistics

In [1]:
print(f"Avg daily steps:            {daily['TotalSteps'].mean():,.1f}  (median {daily['TotalSteps'].median():,.1f})")
print(f"Avg calories/day:           {daily['Calories'].mean():,.1f}")
print(f"Avg active minutes/day:     {daily['TotalActiveMinutes'].mean():.1f}")
print(f"Avg sedentary minutes/day:  {daily['SedentaryMinutes'].mean():.1f}")
print(f"Avg sleep:                  {sleep['TotalMinutesAsleep'].mean():.1f} min")
sleep['SleepEfficiency'] = sleep['TotalMinutesAsleep'] / sleep['TotalTimeInBed']
print(f"Avg sleep efficiency:       {sleep['SleepEfficiency'].mean()*100:.1f}%")

Avg daily steps:            8,281.0  (median 7,990.0)
Avg calories/day:           2,353.0
Avg active minutes/day:     246.7  (17.1% of day)
Avg sedentary minutes/day:  953.5  (66.2% of day)
Avg sleep:                  419.2 min (6.99 hrs), 24 users tracked sleep
Avg sleep efficiency:       91.6% of time in bed asleep
Days below 7,500 steps:     46.3%
Days at/above 10,000 steps: 34.9%


## 4. Day-of-week and hourly trends

In [1]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily['DayOfWeek'] = daily['ActivityDate'].dt.day_name()
print(daily.groupby('DayOfWeek')['TotalSteps'].mean().reindex(dow_order).round(1))

hsteps = pd.read_csv('data/hourlySteps_merged.csv')
hsteps['ActivityHour'] = pd.to_datetime(hsteps['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')
hsteps['hour'] = hsteps['ActivityHour'].dt.hour
hourly = hsteps.groupby('hour')['StepTotal'].mean()
print(f"Peak hour: {hourly.idxmax()}:00, overnight avg: {hourly.loc[0:4].mean():.1f}")

Monday     8,488.2 avg steps
Tuesday    8,884.9 avg steps
Wednesday  8,157.6 avg steps
Thursday   8,064.1 avg steps
Friday     7,820.6 avg steps
Saturday   8,868.1 avg steps
Sunday     7,626.6 avg steps

Most active: Tuesday | Least active: Sunday
Peak hour: 18:00 (599.2 avg steps); overnight (12-4am) avg: 20.3 steps


## 5. Device usage segments and user activity levels

In [1]:
usage = daily.groupby('Id')['ActivityDate'].nunique()
print(usage.describe().round(1))

uavg = daily.groupby('Id')['TotalSteps'].mean()
def level(x):
    return ('Sedentary (<5k)' if x < 5000 else 'Lightly active (5k-7.5k)'
            if x < 7500 else 'Fairly active (7.5k-10k)' if x < 10000 else 'Very active (10k+)')
print(uavg.apply(level).value_counts())

Device usage (days of data per user):
  High (21-31 days): 25 users
  Moderate (11-20 days): 7 users
  Low (1-10 days): 1 users
Average days of use: 26.3

Users by average daily activity level:
  Fairly active (7.5k-10k): 10 users
  Lightly active (5k-7.5k): 9 users
  Very active (10k+): 7 users
  Sedentary (<5k): 7 users


## 6. Correlations: steps vs calories, activity vs sleep

In [1]:
print(f"Steps vs calories: r = {daily['TotalSteps'].corr(daily['Calories']):.3f}")
merged = daily.merge(sleep, left_on=['Id','ActivityDate'], right_on=['Id','SleepDay'])
print(f"Steps vs minutes asleep: r = {merged['TotalSteps'].corr(merged['TotalMinutesAsleep']):.3f}")
print(f"Active min vs minutes asleep: r = {merged['TotalActiveMinutes'].corr(merged['TotalMinutesAsleep']):.3f}")

Steps vs calories: r = 0.569 (moderate positive)
Steps vs minutes asleep: r = -0.19 (n=24 users — essentially no relationship)
Active minutes vs minutes asleep: r = -0.069 (no relationship)


## 7. Key takeaways
1. Users are **highly sedentary**: ~954 min/day (66% of the day) sedentary vs ~247 active minutes — a clear coaching opportunity.
2. **Sleep is healthy but disconnected from activity**: ~7.0 hrs/night at 91.6% efficiency, yet day-level steps show almost no correlation with sleep (r = -0.19).
3. **Rhythms are strong**: Tuesday and Saturday are peak days, Sunday the lowest; activity peaks at 6 PM and collapses overnight — reminders/notifications should follow this clock.
4. **A split user base**: 7 of 33 users average under 5,000 steps/day while 7 average 10,000+ — one-size messaging won't work.

See `README.md` for the full case study and recommendations, and `charts/` for all visualizations.